# Create Building Mask

This notebook creates a binary raster mask from a building footprint vector layer using the spatial reference, extent, and resolution of the processed WorldView-3 imagery.

Pixels corresponding to building footprints are assigned a value of 1, while all other pixels are assigned a value of 0.

### Input

- Building footprint vector layer (Shapefile or other vector format supported by GeoPandas)
- Georeferenced WorldView-3 mosaic (`VNIR_SWIR_stack.tif`)

### Output

- Building mask (`Building_mask.tif`)

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import rasterize


# Input and output files

data_dir = Path("../output")
vector_dir = Path("../data")

reference_raster = data_dir / "VNIR_SWIR_stack.tif"
building_vector = vector_dir / "building_footprints.shp"

mask_path = data_dir / "Building_mask.tif"
inverse_mask_path = data_dir / "Building_mask_inverse.tif"
inverse_mask_npy = data_dir / "Building_mask_inverse.npy"


# Load building footprints and reference raster

buildings = gpd.read_file(building_vector)

with rasterio.open(reference_raster) as src:
    transform = src.transform
    crs = src.crs
    height = src.height
    width = src.width

dtype = rasterio.uint8


# Rasterize building footprints

mask = rasterize(
    [(geom, 1) for geom in buildings.geometry],
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype=dtype,
)


# Save building mask

with rasterio.open(
    mask_path,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype=dtype,
    crs=crs,
    transform=transform,
) as dst:
    dst.write(mask, 1)

print(f"Building mask saved as:\n{mask_path.name}")